# Zed – trénink modelu Zeds World

QLoRA fine-tune **Qwen3 4B** na lore Zeds World a GDScript.

**Runtime → Change runtime type → T4 GPU** (free Colab stačí).

Postup: spusť buňky odshora dolů. Trvá cca 1–2 h.


## 1. Instalace

In [ ]:
!pip install -q unsloth
!pip install -q --no-deps --upgrade "trl>=0.15.0" peft accelerate bitsandbytes
import torch; print(torch.cuda.get_device_name(0))

## 2. Dataset

Nahraj `dataset.jsonl` (vygenerovaný `build_dataset.py`) přes panel Files vlevo,
nebo ho stáhni přímo z repozitáře.

In [ ]:
# Varianta A – stáhnout z GitHubu (uprav branch, pokud je jiná):
# !wget -q https://raw.githubusercontent.com/JACKHACKER2021/Project/ccr-4ad76bcf-mf7gfj/zedsworld_ai/data/dataset.jsonl
# Varianta B – vygenerovat přímo tady:
# !git clone -q --depth 1 -b ccr-4ad76bcf-mf7gfj https://github.com/JACKHACKER2021/Project.git
# !cd Project && python3 zedsworld_ai/data/build_dataset.py && cp zedsworld_ai/data/dataset.jsonl ..

from datasets import load_dataset
ds = load_dataset("json", data_files="dataset.jsonl", split="train")
print(ds)
print(ds[0]["messages"][1]["content"][:200])

## 3. Model

Qwen3 4B v 4bitu. Pro 8 GB telefon je to správná velikost –
po kvantizaci na Q4_K_M vyjde výsledek na ~2,5 GB.

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ = 2048   # delší = víc VRAM; 2048 pokryje i delší lore odpovědi

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length = MAX_SEQ,
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = 32,          # kapacita adaptéru; 32 je dost na tento objem dat
    lora_alpha     = 32,
    lora_dropout   = 0.05,
    bias           = "none",
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state   = 1337,
)

## 4. Formátování do chat šablony

In [ ]:
def to_text(batch):
    return {"text": [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in batch["messages"]
    ]}

ds_fmt = ds.map(to_text, batched=True, remove_columns=ds.column_names)
print(ds_fmt[0]["text"][:600])

## 5. Trénink

`num_train_epochs=3` na ~480 příkladech je rozumný základ.
Loss by měl klesat plynule; když spadne pod ~0.3, model se začíná učit data
nazpaměť místo stylu – tam je čas skončit.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = ds_fmt,
    args = SFTConfig(
        output_dir                  = "out",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,     # efektivní batch 8
        num_train_epochs            = 3,
        learning_rate               = 2e-4,
        warmup_ratio                = 0.05,
        lr_scheduler_type           = "cosine",
        logging_steps               = 5,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        max_seq_length              = MAX_SEQ,
        dataset_text_field          = "text",
        report_to                   = "none",
        seed                        = 1337,
    ),
)
stats = trainer.train()

## 6. Zkouška – odpovídá jako Zed?

In [ ]:
FastLanguageModel.for_inference(model)

SYSTEM = ds[0]["messages"][0]["content"]

def zed(prompt, max_new_tokens=400):
    msgs = [{"role":"system","content":SYSTEM},{"role":"user","content":prompt}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True,
                                        return_tensors="pt").to("cuda")
    out = model.generate(ids, max_new_tokens=max_new_tokens, temperature=0.8,
                         top_p=0.9, do_sample=True)
    print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

zed("Kdo je Zéd Zédovský?")
print("\n" + "="*60 + "\n")
zed("Napiš drsnou scénu, jak Smrtokostrovlak sráží auto Jasdayovy rodiny.")
print("\n" + "="*60 + "\n")
zed("Napiš v GDScriptu funkci, která přepíná kamery a ubírá energii.")

## 7. Export do GGUF

Merge adaptéru do modelu a kvantizace na Q4_K_M – to je ten soubor,
který pojede v telefonu.

In [ ]:
model.save_pretrained_gguf("zed-4b", tokenizer, quantization_method="q4_k_m")
!ls -lh zed-4b/*.gguf

## 8. Stažení

Soubor je ~2,5 GB. Přes Google Drive to je spolehlivější než přímé stažení.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp zed-4b/*Q4_K_M*.gguf /content/drive/MyDrive/zed-4b-q4_k_m.gguf
!ls -lh /content/drive/MyDrive/zed-4b-q4_k_m.gguf

---

## Dál

GGUF soubor jde rovnou do Android aplikace (fáze 4) nebo se dá
hned vyzkoušet v desktopové aplikaci jako LM Studio / Jan.

**Ladění, když výsledek nesedí:**

| problém | co udělat |
|---|---|
| Neumí lore, plete si fakta | víc lore příkladů v `lore_zedsworld.py`, zvýšit `LORE_REPEAT` |
| Opakuje věty nazpaměť | méně epoch (2), nižší `learning_rate` (1e-4) |
| Zapomněl češtinu / drmolí | `r` na 16, méně epoch – adaptér přebil základní model |
| Odmítá hororové scény | přidat víc příkladů s tagem `style` |
| GDScript nefunguje v Godotu | zvýšit `GDSCRIPT_CAP`, přidat `MAX_SEQ` na 4096 |
